# HATFormer × AraMS-28k — resumable Kaggle training

**What this notebook does:** clones your repo → downloads data + pretrained HATFormer → checks the GPU →
(optional) 3-minute smoke test → trains with the paper's hyper-parameters, checkpointing so a new session can
**resume** → draws LR / loss / accuracy / CER curves → evaluates on the test books.

**Before you start (Kaggle sidebar → Session options):**
1. Accelerator = **GPU T4 x2** (one GPU is used; avoid P100 — new PyTorch builds may not support it)
2. **Internet = ON** (needed for git clone + downloads; requires a phone-verified account)
3. Persistence can stay "No persistence" — outputs of *Save Version* are kept regardless.

**Run order:** first run with `SMOKE_TEST = True` (interactive, ~5 min). When it prints `SMOKE TEST PASSED`,
set `SMOKE_TEST = False`, then **Save Version → Save & Run All (Commit)** and close the tab.
To continue after a stop, see the *RESUME* note in the last cell.

In [ ]:
# ============================ 1. SETTINGS (edit here) ============================
import os, sys, time, json, glob, shutil, subprocess

SESSION_START_TS = time.time()                 # the 12 h Kaggle clock starts ~now
os.environ["SESSION_START_TS"] = str(SESSION_START_TS)

SMOKE_TEST       = True    # True  = ~5 min end-to-end test (train, stop, resume).  False = real run.
EXPERIMENT       = "muharaf_ours"              # paper baseline: Muharaf-pretrained init (use "synth_ours" for the ablation)
TIME_BUDGET_HRS  = 11.0    # stop cleanly + checkpoint after this many hours of *session* time (Kaggle limit is 12)

REPO_URL    = "https://github.com/AMYasserF/arams28k-baseline.git"
REPO_BRANCH = "main"

# --- where the big files come from (skipped automatically if you attached them as Kaggle Datasets) ---
DATASET_ZIP_URL   = "https://zenodo.org/records/21499649/files/AraMS-28k-HTR.zip?download=1"   # AraMS-28k-HTR (Zenodo, 2.4 GB)
HATFORMER_ZIP_URL = "https://zenodo.org/records/14165756/files/HATFormer.zip?download=1"      # official HATFormer code+weights (Zenodo, 1.3 GB; the link in the HATFormer paper)
TOKENIZER_URL     = ""   # optional direct URL of tokenizer.json (leave empty to take it from the Hugging Face repo below)
TOKENIZER_HF_REPO = "Archatext/hatformer-arams28k"   # public repo that ships the 50,265-token HATFormer tokenizer.json
INIT_HF_REPO      = ""   # PIPELINE TEST ONLY: set to "Archatext/hatformer-arams28k" to use that (already AraMS-fine-tuned)
                         # model as init when you do not have the Muharaf-pretrained HATFormer. NOT the paper protocol.
INIT_PATH_OVERRIDE = ""  # only if auto-detection of the pretrained folder fails: full path to hatformer-muharaf
EXTRA_ARGS = ""          # optional trainer flags, e.g. "--batch-size 4 --grad-accum 2" if you ever hit CUDA out-of-memory

WORK = "/kaggle/working"
TMP  = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp/kaggle_temp"   # NOT saved with the version (good: big files)
os.makedirs(TMP, exist_ok=True)
REPO_DIR = f"{WORK}/arams28k-baseline"
RUN_DIR  = f"{WORK}/{'smoke_run' if SMOKE_TEST else 'hatformer_run'}"
SCRIPT   = f"{REPO_DIR}/scripts/models/hatformer/train_hatformer.py"

def sh(cmd, check=True, cwd=None):
    """Run a shell command and stream its output live into the notebook log."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", cwd=cwd, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}")
    return rc

def find_files(root, name, max_depth=7):
    hits = []
    if not os.path.isdir(root): return hits
    base = root.rstrip("/").count("/")
    for dp, dn, fn in os.walk(root):
        if dp.count("/") - base >= max_depth: dn[:] = []
        if name in fn: hits.append(os.path.join(dp, name))
    return sorted(hits)

print("mode:", "SMOKE TEST" if SMOKE_TEST else "FULL RUN", "| run dir:", RUN_DIR)

In [ ]:
# ============================ 2. GPU CHECK (fails fast if no GPU) ============================
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), ("NO GPU! Sidebar -> Session options -> Accelerator -> 'GPU T4 x2', "
                                   "then restart the session and run again.")
x = torch.randn(2048, 2048, device="cuda"); torch.cuda.synchronize(); t=time.time()
for _ in range(20): y = x @ x
torch.cuda.synchronize()
print(f"GPU OK: {torch.cuda.get_device_name(0)} | 20 matmuls (2048^2) in {time.time()-t:.2f}s | "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
del x, y

In [ ]:
# ============================ 3. CLONE / UPDATE YOUR REPO ============================
if os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git -C {REPO_DIR} fetch origin {REPO_BRANCH} && git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")
else:
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
sh("git log -1 --format='repo commit: %h  %s  (%cr)'", cwd=REPO_DIR)
assert os.path.isfile(f"{REPO_DIR}/scripts/models/hatformer/hat_utils.py"), (
    "Your GitHub repo does not contain the new trainer yet. Copy the files from the fixed package "
    "(scripts/models/hatformer/*.py, configs/ocr_hatformer.yaml, kaggle/) into your repo, git push, rerun this cell.")
sh("python -c 'import yaml, matplotlib, pandas, PIL' && echo python deps OK || pip install -q pyyaml matplotlib pandas pillow")

In [ ]:
# ============================ 4. DATA + PRETRAINED WEIGHTS + TOKENIZER ============================
# --- 4a. AraMS-28k-HTR (line crops + metadata.csv) ---
def locate_data():
    for root in ("/kaggle/input", f"{TMP}/data"):
        for m in find_files(root, "metadata.csv"):
            if os.path.isdir(os.path.join(os.path.dirname(m), "images")):
                return os.path.dirname(m)
    return None

DATA_DIR = locate_data()
if DATA_DIR is None:
    assert DATASET_ZIP_URL, "No dataset found. Attach AraMS-28k-HTR as a Kaggle Dataset or set DATASET_ZIP_URL."
    os.makedirs(f"{TMP}/data", exist_ok=True)
    sh(f"wget -nv -c -O {TMP}/AraMS-28k-HTR.zip '{DATASET_ZIP_URL}'")
    sh(f"unzip -q -o {TMP}/AraMS-28k-HTR.zip -d {TMP}/data && rm -f {TMP}/AraMS-28k-HTR.zip")
    DATA_DIR = locate_data()
assert DATA_DIR, "metadata.csv with an images/ folder next to it was not found - check the zip layout."
n_img = sum(1 for _ in os.scandir(os.path.join(DATA_DIR, "images")))
print(f"DATA_DIR = {DATA_DIR}  ({n_img} files in images/)")

# --- 4b. pretrained HATFormer (Muharaf stage) ---
def model_dirs(root):
    out = []
    for cfgp in find_files(root, "config.json"):
        d = os.path.dirname(cfgp)
        fs = os.listdir(d)
        if any(f.endswith((".safetensors", ".bin")) for f in fs):
            out.append(d)
    return out

def pick_init(dirs, want):
    pref = [d for d in dirs if want in d.lower()]
    return pref[0] if pref else None

want = "muharaf" if EXPERIMENT == "muharaf_ours" else "synth"
if INIT_PATH_OVERRIDE:
    INIT_PATH = INIT_PATH_OVERRIDE
elif INIT_HF_REPO:
    from huggingface_hub import snapshot_download
    INIT_PATH = snapshot_download(INIT_HF_REPO, local_dir=f"{TMP}/init_hf")
    print("!! USING", INIT_HF_REPO, "AS INIT - fine for testing the pipeline, NOT a paper-faithful baseline !!")
else:
    INIT_PATH = pick_init(model_dirs("/kaggle/input") + model_dirs(f"{TMP}/hatformer"), want)
    if INIT_PATH is None:
        assert HATFORMER_ZIP_URL, ("Muharaf-pretrained HATFormer not found: attach it as a Kaggle Dataset, set HATFORMER_ZIP_URL, "
                                   "or (pipeline test only) set INIT_HF_REPO.")
        os.makedirs(f"{TMP}/hatformer", exist_ok=True)
        sh(f"wget -nv -c -O {TMP}/HATFormer.zip '{HATFORMER_ZIP_URL}'")
        sh(f"unzip -q -o {TMP}/HATFormer.zip -d {TMP}/hatformer && rm -f {TMP}/HATFormer.zip")
        found = model_dirs(f"{TMP}/hatformer")
        INIT_PATH = pick_init(found, want)
        assert INIT_PATH, f"No folder with '{want}' in its path among: {found}. Set INIT_PATH_OVERRIDE."
print("INIT_PATH =", INIT_PATH, "|", sorted(os.listdir(INIT_PATH)))

# --- 4c. tokenizer.json ---
TOK_DST = f"{REPO_DIR}/scripts/models/hatformer/tokenizer/tokenizer.json"
if not os.path.isfile(TOK_DST) or os.path.getsize(TOK_DST) < 100_000:
    os.makedirs(os.path.dirname(TOK_DST), exist_ok=True)
    cands = (find_files("/kaggle/input", "tokenizer.json") + find_files(f"{TMP}/hatformer", "tokenizer.json")
             + find_files(INIT_PATH, "tokenizer.json", 1))
    cands = [c for c in cands if os.path.getsize(c) > 100_000]
    if cands:
        shutil.copy(cands[0], TOK_DST)
    elif TOKENIZER_URL:
        sh(f"wget -nv -O {TOK_DST} '{TOKENIZER_URL}'")
    else:
        from huggingface_hub import hf_hub_download
        shutil.copy(hf_hub_download(TOKENIZER_HF_REPO, "tokenizer.json", local_dir=f"{TMP}/tok_hf"), TOK_DST)
print("tokenizer.json:", TOK_DST, f"{os.path.getsize(TOK_DST)/1e6:.2f} MB")

COMMON = (f"--experiment {EXPERIMENT} --data-dir '{DATA_DIR}' --output-dir '{RUN_DIR}' "
          f"--init-path '{INIT_PATH}' --tokenizer '{TOK_DST}' {EXTRA_ARGS}")

In [ ]:
# ============================ 5. PREFLIGHT (≈1 min): proves everything is wired correctly ============================
sh(f"python -u {SCRIPT} {COMMON} --preflight --limit 200")

In [ ]:
# ============================ 6. TRAIN ============================
if SMOKE_TEST:
    # tiny end-to-end test: train 30 steps -> stop -> RESUME -> continue to 60.  Uses its own folder.
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    small = f"--limit 600 --val-every 15 --ckpt-every 15 --val-max-lines 32 --warmup-steps 10 --time-budget-hours 0 --resume '{RUN_DIR}'"
    sh(f"python -u {SCRIPT} {COMMON} --max-steps 30 {small}")
    print("\n" + "="*20, "second process: must say [RESUME] and continue from step 30", "="*20)
    sh(f"python -u {SCRIPT} {COMMON} --max-steps 60 {small}")
    st = json.load(open(f"{RUN_DIR}/status.json"))
    assert st["finished"] and st["step"] == 60, st
    assert os.path.isfile(f"{RUN_DIR}/plots/training_curves.png")

    # --- SMOKE TEST PREDICTION EXPORT VERIFICATION ---
    import pandas as pd
    smoke_preds_dir = f"{RUN_DIR}/smoke_preds"
    shutil.rmtree(smoke_preds_dir, ignore_errors=True)
    sh(f"python -u {SCRIPT} {COMMON} --eval-only --split test --limit 32 --num-beams 1 --checkpoint '{RUN_DIR}/best' --save-preds-dir '{smoke_preds_dir}'")
    smoke_csvs = sorted(glob.glob(f"{smoke_preds_dir}/*.csv"))
    assert len(smoke_csvs) > 0, f"No prediction CSVs found in {smoke_preds_dir}"
    for pcsv in smoke_csvs:
        df_smoke = pd.read_csv(pcsv)
        assert len(df_smoke) > 0, f"Prediction CSV {pcsv} is empty"
        for req_col in ["line_uid", "book_id", "gt_normalized", "hatformer_pred"]:
            assert req_col in df_smoke.columns, f"Missing required column '{req_col}' in {pcsv}"
        print(f"[SMOKE TEST] Verified {os.path.basename(pcsv)}: {len(df_smoke)} rows | columns={df_smoke.columns.tolist()}")
    assert os.path.isfile(f"{smoke_preds_dir}/eval_summary.json"), f"eval_summary.json missing from {smoke_preds_dir}"

    print("\nSMOKE TEST PASSED  ->  now set SMOKE_TEST = False, then Save Version -> Save & Run All (Commit).")
else:
    sh(f"python -u {SCRIPT} {COMMON} --resume auto --time-budget-hours {TIME_BUDGET_HRS}")


In [ ]:
# ============================ 7. CURVES: learning rate, loss, accuracy, CER ============================
from IPython.display import Image, display
for name in ("training_curves.png", "lr_curve.png", "loss_curve.png", "accuracy_curve.png"):
    p = f"{RUN_DIR}/plots/{name}"
    if os.path.isfile(p):
        print(name); display(Image(p))
sh(f"tail -n 8 '{RUN_DIR}/metrics.csv' | cut -c1-200", check=False)
status = json.load(open(f"{RUN_DIR}/status.json")) if os.path.isfile(f"{RUN_DIR}/status.json") else {}
print("STATUS:", status)

In [ ]:
# ============================ 8. TEST EVALUATION (only when training has finished) ============================
if SMOKE_TEST:
    print("smoke test: skipping test evaluation")
elif status.get("finished"):
    import pandas as pd
    sh(f"python -u {SCRIPT} {COMMON} --eval-only --split test --num-beams 1 --checkpoint '{RUN_DIR}/best' --save-preds-dir '{RUN_DIR}/predictions_beams1'")
    sh(f"python -u {SCRIPT} {COMMON} --eval-only --split test --num-beams 3 --checkpoint '{RUN_DIR}/best' --save-preds-dir '{RUN_DIR}/predictions_beams3'")

    print("\n" + "="*20, "EXPORTED PREDICTION CSVS SUMMARY", "="*20)
    for bdir in (f"{RUN_DIR}/predictions_beams1", f"{RUN_DIR}/predictions_beams3"):
        print(f"\nDirectory: {bdir}")
        for csv_f in sorted(glob.glob(f"{bdir}/*.csv")):
            df_c = pd.read_csv(csv_f)
            print(f"  {os.path.basename(csv_f)}: {len(df_c):5d} lines | columns={df_c.columns.tolist()}")
        sum_f = f"{bdir}/eval_summary.json"
        if os.path.isfile(sum_f):
            with open(sum_f, encoding="utf-8") as f:
                print("  eval_summary.json:", f.read().strip())
else:
    print(f"Training stopped early at step {status.get('step')} ({status.get('reason')}). "
          "Skipping test evaluation - resume first (see next cell).")


In [ ]:
# ============================ 9. WRAP-UP ============================
if "status" not in globals():
    status = json.load(open(f"{RUN_DIR}/status.json")) if os.path.isfile(f"{RUN_DIR}/status.json") else {}

if not SMOKE_TEST and status.get("finished") and os.path.isdir(f"{RUN_DIR}/best"):
    best_dir = f"{RUN_DIR}/best"
    tok_best = f"{best_dir}/tokenizer.json"
    if os.path.isfile(TOK_DST):
        shutil.copy2(TOK_DST, tok_best)
        print(f"[WRAP-UP] Copied tokenizer: {TOK_DST} -> {tok_best}")
    
    zip_dst_base = f"{WORK}/hatformer_best_weights"
    zip_path = shutil.make_archive(zip_dst_base, "zip", root_dir=best_dir)
    print(f"[WRAP-UP] Created self-contained archive: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

if SMOKE_TEST:
    shutil.rmtree(RUN_DIR, ignore_errors=True)          # don't leave smoke files in the saved output

sh(f"du -sh {WORK}/* 2>/dev/null | sort -h | tail", check=False)
print('''
HOW TO RESUME IN A NEW SESSION (when STATUS above says finished: false)
  1. Wait for this version to show "Complete" (Versions panel).
  2. Open the notebook in the editor -> right sidebar -> Input -> "+ Add Input" ->
     choose "Notebooks"/"Your Work" -> select THIS notebook (its latest version output).
     (Alternative: open the version's Output tab -> "New Dataset", then attach that dataset.)
  3. Keep SMOKE_TEST = False, Save Version -> Save & Run All (Commit).
  The trainer finds ckpt_last.pt under /kaggle/input automatically and logs a [RESUME] line.
''')
